Importação das bibliotecas:

In [1]:
import pandas as pd
import unicodedata

Carregamento do Dataset:

In [2]:
arquivos = ["focos_br_ref_2020.csv", "focos_br_ref_2021.csv", "focos_br_ref_2022.csv", "focos_br_ref_2023.csv", "focos_br_ref_2024.csv"]

df_anuais = [pd.read_csv(arq) for arq in arquivos]

df = pd.concat(df_anuais, ignore_index=True)

print(df.head())

       id_bdq                               foco_id     lat     lon  \
0  1422833677  af379e6c-9202-354f-8ef1-d1eec5a371e4  -3.650 -54.732   
1  1422833676  f3f9fabe-661e-3e2a-a641-34db5339e733  -4.128 -47.074   
2  1422833578  b47fec02-9489-3696-9f0c-1afd415e080e -10.513 -53.483   
3  1422833579  4a7ac6de-0723-395c-88d2-002ad439edb8 -10.504 -53.498   
4  1422833580  59553bac-9736-34e7-9124-937a239e040d -10.494 -46.488   

              data_pas    pais       estado           municipio     bioma  
0  2020-06-16 17:00:00  Brasil         PARÁ              PLACAS  Amazônia  
1  2020-06-16 17:00:00  Brasil     MARANHÃO  ITINGA DO MARANHÃO  Amazônia  
2  2020-06-16 16:55:00  Brasil  MATO GROSSO  PEIXOTO DE AZEVEDO  Amazônia  
3  2020-06-16 16:55:00  Brasil  MATO GROSSO  PEIXOTO DE AZEVEDO  Amazônia  
4  2020-06-16 16:55:00  Brasil    TOCANTINS            MATEIROS   Cerrado  


Quantidade de linhas e colunas do Dataframe concatenado:

In [3]:
lin, col = df.shape

print("-" * 60 + "\n")
print(f"Numero de linhas: {lin}")
print(f"Numero de Colunas: {col}")
print("\n" + "-" * 60)

------------------------------------------------------------

Numero de linhas: 1075841
Numero de Colunas: 9

------------------------------------------------------------


Verficação de Dados Faltantes:

In [4]:
print("-" * 60 + "\n")
print(f"Valores faltantes por coluna:")
print(df.isna().sum())
print("\n" + "-" * 60)

------------------------------------------------------------

Valores faltantes por coluna:
id_bdq       0
foco_id      0
lat          0
lon          0
data_pas     0
pais         0
estado       0
municipio    0
bioma        0
dtype: int64

------------------------------------------------------------


Correção de Inconsistências:

In [5]:
# Remocao de duplicatas (caso exista)
antes = len(df)
df = df.drop_duplicates(subset=['id_bdq'])
print(f"Registros removidos por duplicidade: {antes - len(df)}")

# Remocao de pontos fora do territorio brasileiro, caso exista (latitude entre 5 e 34 | longitude entre 34 e 74)
inconsistentes = df[(df['lat'] > 6) | (df['lat'] < -34) | (df['lon'] > -34) | (df['lon'] < -74)]
print(f"Pontos fora do território brasileiro: {len(inconsistentes)}")

df = df.drop(inconsistentes.index)

# Arredonda Latitude e Longitude em 2 casas decimais (regiao de ~1km), agrupando focos que sao os mesmos, porem detectados com variacoes pequenas de posicao
df['lat_arr'] = df['lat'].round(2)
df['lon_arr'] = df['lon'].round(2)

# Caso o registro tenha mesma data, hora, municipio, latitude e longitude, sera considerado como duplicata
criterio_duplicado = ['data_pas', 'municipio', 'lat_arr', 'lon_arr']

antes = len(df)

df = df.drop_duplicates(subset=criterio_duplicado, keep='first')

# Limpa as colunas de arredondamento para nao sujar o dataset
df = df.drop(columns=['lat_arr', 'lon_arr'])

print(f"Registros 'quase duplicados' removidos: {antes - len(df)}")


Registros removidos por duplicidade: 0
Pontos fora do território brasileiro: 0
Registros 'quase duplicados' removidos: 43718


Padronização de Variáveis:

In [6]:
def normalizar_texto(texto):
    # Converte para string e remove espaços extras nas pontas
    texto = str(texto).strip()
    
    # separacao da letra com o acento
    formato_nfkd = unicodedata.normalize('NFKD', texto)

    # Limpa caracteres que nao sao ASCII
    texto_limpo = formato_nfkd.encode('ASCII', 'ignore').decode('ASCII')
    
    return texto_limpo.upper()

colunas_texto = ['estado', 'municipio', 'bioma']

for col in colunas_texto:
    df[col] = df[col].apply(normalizar_texto)

print(f"Valores apos padronizacao:")
df.head()

Valores apos padronizacao:


,id_bdq,foco_id,lat,lon,data_pas,pais,estado,municipio,bioma
0,1422833677,af379e6c-9202-354f-8ef1-d1eec5a371e4,-3.650,-54.732,2020-06-16 17:00:00,Brasil,PARA,PLACAS,AMAZONIA
1,1422833676,f3f9fabe-661e-3e2a-a641-34db5339e733,-4.128,-47.074,2020-06-16 17:00:00,Brasil,MARANHAO,ITINGA DO MARANHAO,AMAZONIA
2,1422833578,b47fec02-9489-3696-9f0c-1afd415e080e,-10.513,-53.483,2020-06-16 16:55:00,Brasil,MATO GROSSO,PEIXOTO DE AZEVEDO,AMAZONIA
3,1422833579,4a7ac6de-0723-395c-88d2-002ad439edb8,-10.504,-53.498,2020-06-16 16:55:00,Brasil,MATO GROSSO,PEIXOTO DE AZEVEDO,AMAZONIA
4,1422833580,59553bac-9736-34e7-9124-937a239e040d,-10.494,-46.488,2020-06-16 16:55:00,Brasil,TOCANTINS,MATEIROS,CERRADO


Preparação de Dados para Análise:

In [7]:
# converte a coluna data_pas para o formato de data
df['data_pas'] = pd.to_datetime(df['data_pas'])

# criacao de colunas diferentes de data e hora
df['data'] = df['data_pas'].dt.date
df['hora'] = df['data_pas'].dt.time

# criacao de colunas de mes e ano
df['ano'] = df['data_pas'].dt.year
df['mes'] = df['data_pas'].dt.month

# remocao da coluna de pais, visto que sempre tera valor 'Brasil'
df = df.drop(columns=['pais'])

print("Dados desmembrados e preparados!")
df.head()

Dados desmembrados e preparados!


,id_bdq,foco_id,lat,lon,data_pas,estado,municipio,bioma,data,hora,ano,mes
0,1422833677,af379e6c-9202-354f-8ef1-d1eec5a371e4,-3.650,-54.732,2020-06-16 17:00:00,PARA,PLACAS,AMAZONIA,2020-06-16,17:00:00,2020,6
1,1422833676,f3f9fabe-661e-3e2a-a641-34db5339e733,-4.128,-47.074,2020-06-16 17:00:00,MARANHAO,ITINGA DO MARANHAO,AMAZONIA,2020-06-16,17:00:00,2020,6
2,1422833578,b47fec02-9489-3696-9f0c-1afd415e080e,-10.513,-53.483,2020-06-16 16:55:00,MATO GROSSO,PEIXOTO DE AZEVEDO,AMAZONIA,2020-06-16,16:55:00,2020,6
3,1422833579,4a7ac6de-0723-395c-88d2-002ad439edb8,-10.504,-53.498,2020-06-16 16:55:00,MATO GROSSO,PEIXOTO DE AZEVEDO,AMAZONIA,2020-06-16,16:55:00,2020,6
4,1422833580,59553bac-9736-34e7-9124-937a239e040d,-10.494,-46.488,2020-06-16 16:55:00,TOCANTINS,MATEIROS,CERRADO,2020-06-16,16:55:00,2020,6
